In [1]:
HEIGHT = 21
WIDTH = 28
FRAMES = 1000
FPS = 1

In [7]:
from query_generator import generate_queries_by_frame
generate_queries_by_frame(HEIGHT, WIDTH, FRAMES, FPS)

Done!                                                  


In [ ]:
!. align.sh

Output for queries/query_0001.fa already found, skipping
Output for queries/query_0025.fa already found, skipping
Output for queries/query_0049.fa already found, skipping
Output for queries/query_0073.fa already found, skipping
Output for queries/query_0097.fa already found, skipping
Output for queries/query_0121.fa already found, skipping
Output for queries/query_0145.fa already found, skipping
Aligning to queries/query_0169.fa
Aligning to queries/query_0193.fa
Aligning to queries/query_0217.fa


In [ ]:
from output_parser import parse_to_json_files
parse_to_json_files()

In [ ]:
import json

FRAMES = 170

data = {}
for i in range(1, FRAMES, 24//FPS):
    with open(f"outputs/fasta_{i:04d}.json", "r") as f:
        data[i] = json.loads(f.read())

# data = {}
# for x in range(WIDTH):
#     for y in range(HEIGHT):
#         with open(f"outputs/fasta_{y:02d}_{x:02d}.json", "r") as f:
#             data[(x, y)] = json.loads(f.read())

# data = {}
# for x in range(WIDTH):
#     for y in range(HEIGHT):
#         with open(f"queries/query_{y:02d}_{x:02d}.fa", "r") as f:
#             data[(x, y)] = {
#                 "name": "Test",
#                 "status": "Sequence found",
#                 "sequence": f.read().split('\n')[1]
#             }

print(len(data), "files loaded")

In [ ]:
def get_sequence(coord):
    if coord["status"] != "Sequence found":
        return ""
    s = coord["sequence"]
    if coord["rev-comp"]:
        return s[::-1].replace("A", "_").replace("T", "A").replace("_", "T").replace("C", "_").replace("G", "C").replace("_", "G")
    return s

# sequences = {}
# for x in range(WIDTH):
#     for y in range(HEIGHT):
#         sequences[(x, y)] = get_sequence(data[(x, y)])
        
sequences = {}
n = 0
for i in range(1, FRAMES, 24//FPS):
    sequences[n] = get_sequence(data[i])
    n += 1

print(sequences[0])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

plt.rcParams['font.family'] = 'STIXGeneral'

fig, ax = plt.subplots()
ax.axis("off")

fig_text = {}
for x in range(WIDTH):
    for y in range(HEIGHT):
        fig_text[(x, y)] = fig.text(x/WIDTH, 1-y/HEIGHT, "")
    
def draw(i):
#     for x in range(WIDTH):
#         for y in range(HEIGHT):
#             sequence = sequences[(x, y)]
#             if sequence != "":
#                 base = sequence[i]
#                 color = (1,0,0) if base in "CT" else (0,0,1)
#                 fig_text[(x, y)].set_text(base)
#                 fig_text[(x, y)].set_color(color)
    if sequences[i] == "":
        return
    sequence = np.array(list(sequences[i])).reshape((HEIGHT, WIDTH))
    for x in range(WIDTH):
        for y in range(HEIGHT):
            base = sequence[y,x]
            color = (1,0,0) if base in "CT" else (0,0,1)
            fig_text[(x, y)].set_text(base)
            fig_text[(x, y)].set_color(color)

ani = animation.FuncAnimation(fig=fig, func=draw, frames=len(sequences), interval=1)

FFwriter = animation.FFMpegWriter(fps=FPS)
ani.save('animation.mp4', writer=FFwriter)

In [ ]:
from IPython.display import Video
Video("animation.mp4")